# ESP32-LLM Training Pipeline on Google Colab
Train Quantization-Aware Ternary (1.58-bit) LLM models (`Micro-LM-Pico`, `Micro-LM-Ultra`, `Micro-LM-S3-Large`) and export checkpoints & C binaries directly to Google Drive.

In [ ]:
# 1. Mount Google Drive & Set Sync Destination
import os
from google.colab import drive

drive.mount('/content/drive')

GDRIVE_OUT_DIR = '/content/drive/MyDrive/esp32_llm_checkpoints'
os.makedirs(GDRIVE_OUT_DIR, exist_ok=True)
print(f'Google Drive mounted! Checkpoints will sync to: {GDRIVE_OUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted! Checkpoints will sync to: /content/drive/MyDrive/esp32_llm_checkpoints


In [ ]:
# 2. Check GPU & Install Dependencies
!nvidia-smi
!pip install -q datasets tqdm torch

Sat Aug 15 09:01:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 3. Setup Project Code & Clone / Sync Repository
import os
%cd /content
if not os.path.exists('micro-lm'):
    !git clone https://github.com/adesgautam/micro-lm.git micro-lm
%cd /content/micro-lm
!git pull

/content
Cloning into 'micro-lm'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 183 (delta 14), reused 34 (delta 12), pack-reused 124 (from 1)
Receiving objects: 100% (183/183), 79.33 MiB | 777.00 KiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/micro-lm
Already up to date.


In [ ]:
# 4. Prepare Corpus & Dataset (Download HuggingFace Lyrics if needed)
import os

if not os.path.exists('datasets/raw/lyrics_corpus.txt'):
    print('Preparing dataset from HuggingFace...')
    !python scripts/prepare_hf_lyrics.py
    if os.path.exists('datasets/raw/lyrics_corpus_v2.txt'):
        !cp datasets/raw/lyrics_corpus_v2.txt datasets/raw/lyrics_corpus.txt

print('Dataset ready!')

Preparing dataset from HuggingFace...
Loading dataset from HuggingFace...

song_lyrics 2.csv: downloading bytes:   0% 1.38k/9.07G [00:02<3870:24:20, 651B/s]
song_lyrics 2.csv: downloading bytes:   0% 8.45M/9.07G [00:02<28:16, 5.34MB/s,   131B/s  ]
song_lyrics 2.csv: downloading bytes:   0% 27.7M/9.07G [00:02<07:53, 19.1MB/s, 2.17MB/s  ] ]
song_lyrics 2.csv: downloading bytes:   0% 36.2M/9.07G [00:02<06:14, 24.1MB/s, 2.85MB/s  ] ]
song_lyrics 2.csv: reconstructing file:   1% 48.0M/9.07G [00:02<04:46, 31.5MB/s, 2.15MB/s  ]
song_lyrics 2.csv: downloading bytes:   1% 52.7M/9.07G [00:03<04:54, 30.6MB/s, 4.38MB/s  ] ]
song_lyrics 2.csv: downloading bytes:   1% 58.8M/9.07G [00:03<04:15, 35.2MB/s, 4.71MB/s  ] ]
song_lyrics 2.csv: downloading bytes:   1% 71.2M/9.07G [00:03<03:28, 43.3MB/s, 5.60MB/s  ] ]
song_lyrics 2.csv: downloading bytes:   1% 79.9M/9.07G [00:03<03:27, 43.3MB/s, 6.59MB/s  ]] 
song_lyrics 2.csv: downloading bytes:   1% 90.6M/9.07G [00:03<03:13, 46.3MB/s, 7.27MB/s  ]]
song_lyri

In [ ]:
# 5. Define GDrive Sync Helper
import shutil
import glob

def sync_to_gdrive():
    print(f'Syncing checkpoints and binaries to GDrive ({GDRIVE_OUT_DIR})...')
    if os.path.exists('checkpoints'):
        gdrive_ckpt = os.path.join(GDRIVE_OUT_DIR, 'checkpoints')
        shutil.copytree('checkpoints', gdrive_ckpt, dirs_exist_ok=True)
    
    gdrive_fw = os.path.join(GDRIVE_OUT_DIR, 'firmware_binaries')
    os.makedirs(gdrive_fw, exist_ok=True)
    for bin_file in glob.glob('firmware/src/*.bin'):
        shutil.copy(bin_file, gdrive_fw)
        
    print(f'Sync complete! Files backed up in {GDRIVE_OUT_DIR}')

sync_to_gdrive()

In [ ]:
# 6. Train Micro-LM-Pico (207K params)
!python scripts/train_qat.py --config micro_lm_pico --epochs 50
sync_to_gdrive()

In [ ]:
# 7. Train Micro-LM-Ultra (181K params)
!python scripts/train_qat.py --config micro_lm_ultra --epochs 50
sync_to_gdrive()

In [ ]:
# 8. Train Micro-LM-S3-Large (ESP32-S3 tuned model)
!python scripts/train_qat.py --config micro_lm_s3_large --epochs 30
sync_to_gdrive()

In [ ]:
# 9. Final Sync Summary
sync_to_gdrive()
print('ALL TRAINING JOBS AND EXPORTS COMPLETED AND SAVED TO GDRIVE!')